<div align='center'>

# Práctica 6 - Spark

<img src='https://media1.giphy.com/media/v1.Y2lkPTc5MGI3NjExcWdhaHp4cG5qcWdqYWNjYnpqdmRiNDVlZjV3NXF6ODZpOTJmdHh4aSZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/BmmfETghGOPrW/giphy.gif'>

</div>


---

## 1)

Indique (pensando en lo que hace cada operación) si las siguientes transformaciones son *narrow* o *wide*.


| Transformación       | Tipo       | Justificación breve                                                                  |
| -------------------- | ---------- | ------------------------------------------------------------------------------------ |
| **a) cartesian**     | **wide**   | Requiere combinar todas las particiones de ambos RDDs → gran transferencia de datos. |
| **b) coalesce**      | **narrow** | Reorganiza particiones sin mover datos entre nodos (salvo si `shuffle=True`).        |
| **c) distinct**      | **wide**   | Necesita agrupar y eliminar duplicados mediante un shuffle.                          |
| **d) flatMap**       | **narrow** | Cada partición procesa sus elementos sin comunicación entre nodos.                   |
| **e) flatMapValues** | **narrow** | Aplica `flatMap` solo a los valores, sin requerir redistribución de claves.          |
| **f) intersection**  | **wide**   | Requiere comparar datos entre particiones de ambos RDDs (shuffle).                   |
| **g) repartition**   | **wide**   | Cambia la distribución de datos entre particiones, forzando shuffle.                 |
| **h) subtract**      | **wide**   | Compara y resta datos entre RDDs distintos, requiere movimiento de datos.            |
| **i) union**         | **narrow** | Simplemente concatena RDDs sin reordenar (no requiere shuffle).                      |

---


---

## 2)

Usando el dataset **EstacionesMeteorológicas**, imprima el ID de la estación que tiene el **máximo registro de humedad**, el **ID de la estación con máximo registro en temperatura** y el **ID de la estación con el máximo registro de precipitación** usando solo **seis transformaciones**, incluyendo la transformación `textFile`.

---


In [1]:
from pyspark.sql import SparkSession
import os, sys, glob

# ===============================
# CONFIGURACIÓN LOCAL SPARK
# ===============================
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("EstacionesMeteorologicasLocal") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

# ===============================
# RUTAS LOCALES
# ===============================
inputDir = r"C:\Users\Fabian\Desktop\big-data\Datasets_spark\EstacionesMeteorologicas"

# ===============================
# FUNCIONES AUXILIARES
# ===============================
def splitter(line):
    parts = line.strip().split('\t')
    return (parts[0], parts[1], float(parts[2]), float(parts[3]), float(parts[4]))

def to_celsius(fahrenheit):
    return (float(fahrenheit) - 32) / 1.8

def to_mm(cm):
    return float(cm) * 10

def splitterNorte(line):
    e = splitter(line)
    return (e[0], e[1], e[2], e[3], e[4])

def splitterSur(line):
    e = splitter(line)
    return (e[0], e[1], to_celsius(e[2]), e[3], to_mm(e[4]))

def fMaximos(e1, e2):
    temp = e1[0] if e1[0][2] > e2[0][2] else e2[0]
    hum  = e1[1] if e1[1][3] > e2[1][3] else e2[1]
    prec = e1[2] if e1[2][4] > e2[2][4] else e2[2]
    return (temp, hum, prec)

# ===============================
# FUNCIONES DE CARGA
# ===============================
def load_folder(path, splitter_func):
    """Carga todos los .txt de una carpeta como un único RDD"""
    files = glob.glob(os.path.join(path, "*.txt"))
    if not files:
        raise FileNotFoundError(f"No se encontraron archivos en: {path}")
    rdds = [sc.textFile(f).map(splitter_func) for f in files]
    return sc.union(rdds)

# ===============================
# CARGA DE ARCHIVOS
# ===============================
norte = load_folder(os.path.join(inputDir, "Norte"), splitterNorte)   # TR1
sur   = load_folder(os.path.join(inputDir, "Sur"), splitterSur)       # TR2

print(f"Total Norte: {norte.count()} | Total Sur: {sur.count()}")

# ===============================
# TRANSFORMACIONES PRINCIPALES
# ===============================
estaciones = norte.union(sur)                                         # TR3
estaciones = estaciones.map(lambda e: (e, e, e))                      # TR4
maximos = estaciones.reduce(fMaximos)                                 # TR5
ids = (maximos[0][0], maximos[1][0], maximos[2][0])                  # TR6

# ===============================
# RESULTADOS
# ===============================
print("\n=== RESULTADOS ===")
print(f"ID máx temperatura: {ids[0]}")
print(f"ID máx humedad: {ids[1]}")
print(f"ID máx precipitación: {ids[2]}")

# ===============================
# FINALIZACIÓN
# ===============================
spark.stop()
print("\nEjecución completada ✅")


Total Norte: 2009 | Total Sur: 2265

=== RESULTADOS ===
ID máx temperatura: 246
ID máx humedad: 286
ID máx precipitación: 282

Ejecución completada ✅


---

## 3)

Indique en cuántas etapas se ejecutan los siguientes scripts

### a)

```python
A = sc.textFile("Caso B1")
B = A.map(fMap1)
C = B.filter(fFilter1)
C = C.map(fmap2)

A = sc.textFile("Caso B2")
B = A.map(fmap3)
D = C.join(B)
D = D.filter(fFilter2)
final = D.reduce(fReduce1)
```

### b)

```python
A = sc.textFile("Caso C")
B = A.map(fMap1)
C = B.filter(fFilter1)
D = C.groupByKey()
C = D.filter(fFilter2)
E = C.groupByKey()
E = D.cogroup(E)
final = E.reduce(fReduce1)
```

---

### c)

```python
A = sc.textFile("Caso D.1")
B = sc.textFile("Caso D.2")

C = sc.textFile("Caso D.3")
A = A.map(fMap1)
A = A.distinct()
B = B.filter(fFilter1)
C = C.filter(fFilter1)
E = C.join(B)
B = A.map(fMap2)
C = E.map(fMap3)
D = B.union(C)
F = E.map(fMap4)
E = F.filter(fFilter2)
D = D.union(E).union(B)
B = D.subtract(E)
final = B.count()
```


---

## 4)

Utilizando el dataset **Banco**, escriba un script que permita determinar si las siguientes afirmaciones son verdaderas:

a) El banco tiene más clientes **europeos** que **americanos**.

b) El **promedio de edad** de los clientes americanos es **menor** que el de los europeos.

c) Los **americanos deben más plata** que los europeos
(Un cliente debe plata si la suma de montos de todas sus cajas de ahorro es negativa).

d) Los clientes **americanos suelen sacar**, en promedio, **préstamos con mayor cantidad de cuotas** que los europeos.


---

## 5)
Utilizando el dataset **Banco**, escriba un script que permita calcular el **factor de riesgo** de todos sus clientes.  

El factor de riesgo de un cliente se calcula de la siguiente manera:

$$
factorRiesgo = \frac{\left(\frac{D}{E} + 0.001\right)^{F}}{\left(\frac{A}{B}\right)^{\frac{1}{B - C + 1}}}
$$

donde:

- **A**: saldo total entre todas las cajas de ahorro  
- **B**: cantidad de cajas de ahorro  
- **C**: cantidad de cajas de ahorro con saldo negativo  
- **D**: monto total de todos los préstamos  
- **E**: promedio de cuotas entre todos los préstamos  
- **F**: cantidad de préstamos  


## 6)
Realice un script que permita imprimir, **por país**, los **nombres de los clientes** cuyo **factor de riesgo es menor a 2**.
